In [3]:
import pandas as pd
import yfinance as yf
import pyupbit
import requests
from datetime import datetime
from sqlalchemy import create_engine
import pymysql
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 0. MySQL 연결
# ==========================================
db_user = 'root'
db_pw = 'jang1003'
db_host = 'localhost'
db_port = '3306'
db_name = 'quant_db'

db_connection_str = f'mysql+pymysql://{db_user}:{db_pw}@{db_host}:{db_port}/{db_name}'
engine = create_engine(db_connection_str)

# ==========================================
# 1. 가격 데이터 및 환율
# ==========================================
def fetch_current_prices():
    exchange_rate = yf.Ticker("USDKRW=X").history(period="1d")['Close'].iloc[-1]

    prices = {
        'BTC': pyupbit.get_current_price("KRW-BTC"),
        'ETH': pyupbit.get_current_price("KRW-ETH"),
        'VOO': yf.Ticker("VOO").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'GOOGL': yf.Ticker("GOOGL").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'USD': exchange_rate,
        'KRW': 1.0
    }
    return prices, exchange_rate

# ==========================================
# 2.과열 발생 시 리밸런싱 및 신호 생성
# ==========================================
def process_portfolio_data(input_csv):
    df = pd.read_csv(input_csv)
    prices, exchange_rate = fetch_current_prices()

    df['Current_Price'] = df['Asset'].map(prices)
    df['Total_Value_KRW'] = df['Quantity'] * df['Current_Price']
    total_asset_value = df['Total_Value_KRW'].sum()

    # 1) 나스닥, 크립토 탐욕 지수 확인- VIX 지수 가이드: 15 이하(극단적 탐욕/과열), 20~30(주의), 30 이상(공포)
    # 2) 크립도 공포지수는 75이상시 탐욕/과열
    current_vix=18.7
    current_fgi = 55

    print(f"[미국 주식 탐욕 지수] 현재 VIX 지수: {current_vix:.2f} (낮을수록 극단적 탐욕)")
    print(f"[크립토 탐욕 지수] 현재 FGI 지수: {current_fgi:.2f} (높을수록 극단적 탐욕)")

    #시장 과열 시 목표비중의 30% 매도-기본 0%
    us_sell_ratio = 0.0
    crypto_sell_ratio = 0.0

    # 나스닥 이익 실현할 비중 30%(극도의 낙관시장일때)
    if current_vix <= 15:
        us_sell_ratio = 0.30

    #크립토 이익 실현 30%(극도의 낙관 시장일떄)
    if current_fgi >= 75:
        crypto_sell_ratio = 0.30


    # 2) 동적 목표 비중 계산 (판 만큼 달러/원화에 더하기)
    df['Dynamic_Target_Weight'] = df['Target_Weight'].copy()
    us_sold_weight_total = 0
    crypto_sold_weight_total = 0

    for idx, row in df.iterrows():
        asset = row['Asset']
        base_w = row['Target_Weight']

        if asset in ['VOO', 'GOOGL']:
            sell_w = base_w * us_sell_ratio #(목표비중에서 30% 익절할 비율)
            df.at[idx, 'Dynamic_Target_Weight'] = base_w - sell_w #종목 동적 비중 조절
            us_sold_weight_total += sell_w #달러 표시 자산 조절해야 할 총 비중

        elif asset in ['BTC', 'ETH']:
            sell_w = base_w * crypto_sell_ratio
            df.at[idx, 'Dynamic_Target_Weight'] = base_w - sell_w
            crypto_sold_weight_total += sell_w #원화 표시 자산(크립토) 조절해야 할 총 비중

    for idx, row in df.iterrows():
        if row['Asset'] == 'USD':
            df.at[idx, 'Dynamic_Target_Weight'] = row['Target_Weight'] + us_sold_weight_total #확보해야 할 총 달러 비중
        elif row['Asset'] == 'KRW':
            df.at[idx, 'Dynamic_Target_Weight'] = row['Target_Weight'] + crypto_sold_weight_total #확보해야 할 총 원화 비중

    # 3) ±15% 밴드 리밸런싱 매매 신호 생성
    df['Current_Weight_%'] = (df['Total_Value_KRW'] / total_asset_value) * 100 #현재 비중 계산
    df['Weight_Gap_%'] = df['Dynamic_Target_Weight'] - df['Current_Weight_%'] #변경 후 (혹은 변경x) 비중과의 차이

    actions, amt_krw, amt_usd, action_qtys = [], [], [], []
    usd_assets = ['VOO', 'GOOGL']

    for _, row in df.iterrows():
        asset = row['Asset']
        target_w = row['Dynamic_Target_Weight'] #변경된 목표 비중(과열 신호 없으면 그대로 유지)
        current_w = row['Current_Weight_%'] #현재 비중
        current_price = row['Current_Price'] #현재가

        required_krw = total_asset_value * (row['Weight_Gap_%'] / 100)

        if asset in ['USD', 'KRW']:
            exec_krw = 0
            actions.append("CASH_BUFFER")
        else:
            buy_threshold = target_w * 0.85
            sell_threshold = target_w * 1.15

            if current_w <= buy_threshold:
                exec_krw = required_krw
                actions.append("BUY")
            elif current_w >= sell_threshold:
                exec_krw = abs(required_krw)
                actions.append("SELL")
            else:
                exec_krw = 0
                actions.append("HOLD")

        amt_krw.append(exec_krw)
        if asset in usd_assets and exec_krw > 0:
            amt_usd.append(exec_krw / exchange_rate)
        else:
            amt_usd.append(0)
        action_qtys.append(exec_krw / current_price if exec_krw > 0 else 0)

    # 4) 최종 정리 및 출력
    df['Action_Signal'] = actions
    df['Action_Amount_KRW'] = amt_krw
    df['Action_Amount_USD'] = amt_usd
    df['Action_Quantity'] = action_qtys
    df['Last_Updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    print(f"총 자산 가치: {total_asset_value:,.0f} 원")
    print("-" * 115)
    print(f"{'Asset':<10} | {'동적목표%':>6} | {'현재%':>6} | {'상태':<15} | {'주문금액(달러/원)':>20} | {'권장수량'}")
    print("-" * 115)

    for _, row in df.iterrows():
        asset = row['Asset']
        a_krw, a_usd, qty = row['Action_Amount_KRW'], row['Action_Amount_USD'], row['Action_Quantity']
        amt_str = f"${a_usd:,.2f}" if asset in usd_assets and a_usd > 0 else (f"{a_krw:,.0f}원" if a_krw > 0 else "-")
        qty_str = f"{qty:.4f} 개" if qty > 0 else "-"
        print(f"{asset:<10} | {row['Dynamic_Target_Weight']:>7.1f} | {row['Current_Weight_%']:>7.1f} | {row['Action_Signal']:<15} | {amt_str:>20} | {qty_str}")

    df.to_sql(name='portfolio_log', con=engine, if_exists='append', index=False)

    cash_value = df[df['Asset'].isin(['USD', 'KRW'])]['Total_Value_KRW'].sum()
    summary_df = pd.DataFrame({
        'Last_Updated': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
        'Total_Portfolio_Value': [total_asset_value],
        'Cash_Value_KRW': [cash_value],
        'Current_VIX': [current_vix],
        'Current_FGI': [current_fgi]
    })
    summary_df.to_sql(name='portfolio_summary_log', con=engine, if_exists='append', index=False)

    df.to_csv("portfolio_for_tableau.csv", index=False)

process_portfolio_data('my_assets.csv')

[미국 주식 탐욕 지수] 현재 VIX 지수: 18.70 (낮을수록 극단적 탐욕)
[크립토 탐욕 지수] 현재 FGI 지수: 55.00 (높을수록 극단적 탐욕)
총 자산 가치: 32,790,769 원
-------------------------------------------------------------------------------------------------------------------
Asset      |  동적목표% |    현재% | 상태              |           주문금액(달러/원) | 권장수량
-------------------------------------------------------------------------------------------------------------------
BTC        |    50.0 |    50.8 | HOLD            |                    - | -
VOO        |    20.0 |    19.8 | HOLD            |                    - | -
ETH        |    10.0 |    10.5 | HOLD            |                    - | -
GOOGL      |    10.0 |    10.0 | HOLD            |                    - | -
USD        |     7.0 |     8.9 | CASH_BUFFER     |                    - | -
KRW        |     3.0 |     0.0 | CASH_BUFFER     |                    - | -
